# OnTrack quickstart (Google Colab, T4 GPU)

End-to-end: install → build knowledge-base index → train the triage classifier → load MedGemma 1.5 4B (4-bit) → chat → run the RAG vs no-RAG evaluation on the 60-item held-out test set → export the human-scoring CSV.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**.
2. Open https://huggingface.co/google/medgemma-1.5-4b-it while logged in and accept the Health AI Developer Foundations terms (access is gated).
3. Create a Hugging Face *read* token and save it in Colab **Secrets** (key icon on the left) as `HF_TOKEN`, with notebook access on.

OnTrack is an education prototype. It does not diagnose and never gives medication or insulin doses.

## 1. Get the code

In [ ]:
import os
REPO = "https://github.com/Gen3ratorX/ontrack-food.git"
BRANCH = "main"
if not os.path.exists("/content/ontrack-food"):
    !git clone -b {BRANCH} {REPO} /content/ontrack-food
# If the branch is not pushed yet: upload ontrack-food.zip to /content and run
#   !unzip -q /content/ontrack-food.zip -d /content
%cd /content/ontrack-food

In [ ]:
!pip -q install -r requirements-colab.txt
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Hugging Face login (needed for MedGemma and AfriMed-QA)

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception as e:
    print("No Colab secret found, falling back to interactive login:", e)
    login()

## 3. Knowledge base and retrieval (CPU is fine)

In [ ]:
!python scripts/build_wafct_extract.py      # committed extract is reused; FAO download only with --force
!python scripts/build_index.py
!python -m ontrack.eval.retrieval_eval --k 5 | head -3

In [ ]:
from ontrack.rag.retriever import Retriever
retriever = Retriever.from_config()
for h in retriever.search("Is kenkey better than banku for blood sugar?"):
    print(f"{h.score:.3f}", h.chunk_id, "|", h.title)

## 4. Triage classifier (TF-IDF + logistic regression) and the safety gate

In [ ]:
!python -m ontrack.classifier.baseline
from ontrack.classifier.gate import SafetyGate
gate = SafetyGate.from_config()
for q in ["Is waakye okay for breakfast?", "How many units of insulin for fufu?", "My father fainted and is not responding"]:
    print(gate.classify(q), "<-", q)

## 4b. Meal / food glycaemic-impact classifier (CPU, no model needed)
Recognises Ghanaian and West African foods in free text, including local names (dɔkono, bayerɛ, nkontommire…) and typos, then rates the expected blood-sugar impact as **low / medium / high / insufficient_evidence**:
- An **estimated glycaemic load** is given only when a food has both a cited GI (Ghanaian GI studies) and a WAFCT 2019 carbohydrate value.
- Otherwise the rating uses carbohydrate only or the GI class only, and says so.
- Protein, vegetables, legumes and fat from at least two foods lower the rating by one level.
- Every result lists the foods that drove it and its sources.

All numbers are estimates based on assumed portions. The TF-IDF + LR model is a learned comparison, trained on synthetic engine labels, so its CV score shows label leakage.

In [ ]:
from ontrack.meal.engine import analyse_meal
for meal in ["2 balls of kenkey with fried fish and shito",
             "half ball of banku with okro stew and grilled tilapia",
             "jollof rice with fried chicken, salad and a bottle of coke",
             "kokonte and groundnut soup"]:
    print(analyse_meal(meal).to_context(), "\n")

In [ ]:
# Learned comparison model + evaluation (synthetic data; see README for the leakage caveat)
!python scripts/make_meal_dataset.py
!python -m ontrack.meal.learned | tail -60
!python -m ontrack.eval.meal_eval

## 5. Load MedGemma 1.5 4B in 4-bit (about 3–4 GB of GPU memory)

In [ ]:
from ontrack.config import CONFIG
from ontrack.rag.generator import MedGemmaGenerator
gen = MedGemmaGenerator(CONFIG.model_id, load_in_4bit=True, max_new_tokens=350)  # float16 compute on T4
print(gen.model_id, gen.torch_dtype)

## 6. Chat: RAG vs no-RAG

In [ ]:
from ontrack.pipeline import OnTrackAssistant
rag_bot = OnTrackAssistant(gen, retriever, gate, use_rag=True)
base_bot = OnTrackAssistant(gen, None, gate, use_rag=False)
q = "I have type 2 diabetes and I'm about to eat jollof rice with fried chicken and salad. Is this a good meal choice?"
for bot in (rag_bot, base_bot):
    r = bot.answer(q)
    print(f"=== {r.mode} (triage={r.triage_label})\n{r.answer}\n", [s['doc_id'] for s in r.sources])

In [ ]:
print(rag_bot.answer("I'm shaking and sweating, my sugar is 2.6").answer)

In [ ]:
# A meal question: the safety gate runs first, then the meal analysis [M] is added to the RAG context
r = rag_bot.answer("I'm about to eat 2 balls of kenkey with fried fish and shito. How will it affect my sugar?")
print(r.answer); print(r.meal_analysis["label"], r.meal_analysis["drivers"])

## 7. Optional: zero-shot MedGemma triage vs the TF-IDF baseline

In [ ]:
import json
from ontrack.classifier.zero_shot import evaluate_zero_shot
rows = [json.loads(l) for l in open(CONFIG.triage_data)]
rep = evaluate_zero_shot(gen, rows)
print({k: round(v["recall"], 3) for k, v in rep.items() if isinstance(v, dict) and "recall" in v}, "accuracy", round(rep["accuracy"], 3))

## 8. Evaluation on the 60-item held-out test set

Runs both modes (≈120 generations; roughly 20–40 minutes on a T4 depending on answer length) and scores with MedGemma as an LLM judge. Self-judging is biased — treat the judge scores as a quick signal and use the exported CSV for human/clinician scoring.

In [ ]:
!python -m ontrack.eval.split --n-test 60
from pathlib import Path
from ontrack.eval.harness import load_jsonl, run
from ontrack.eval.judge import LLMJudge
items = load_jsonl(CONFIG.splits_dir / "test.jsonl")
summary = run(items, ["rag", "no_rag"], gen, retriever, gate, LLMJudge(gen), Path("artifacts/runs/colab_t4"))

In [ ]:
import pandas as pd
s = summary["judge_scores"]
cols = ["accuracy_pass_rate", "relevance_pass_rate", "cultural_relevance_pass_rate", "safety_pass_rate", "clarity_pass_rate", "critical_unsafe"]
pd.DataFrame({m: {c: s[m][c] for c in cols} for m in s}).T

In [ ]:
# Separate meal set (7 items, not part of the frozen 60-item test split)
!python -m ontrack.eval.harness --split meals --modes rag no_rag --judge llm --out artifacts/runs/colab_t4_meals

## 9. Human scoring
Download `artifacts/runs/colab_t4/human_scoring.csv` (blinded and shuffled; mode hidden), fill in 1–5 scores per `ontrack/eval/rubric.md`, upload it back, then:

In [ ]:
# !python -m ontrack.eval.harness --score-human artifacts/runs/colab_t4/human_scoring.csv
from google.colab import files
files.download("artifacts/runs/colab_t4/human_scoring.csv")

## 10. Optional: pull diabetes questions from AfriMed-QA v2 for curation (gated dataset, CC BY-SA 4.0)

In [ ]:
!python -m ontrack.eval.afrimedqa --out data/eval/afrimedqa_diabetes_candidates.jsonl
!head -c 1500 data/eval/afrimedqa_diabetes_candidates.jsonl